In [ ]:
#@title Install
%%capture
!pip install gradio openai -q

In [ ]:
#@title API Key
import os
from getpass import getpass
k = getpass("OpenAI API key: ")
if k.strip(): os.environ["OPENAI_API_KEY"] = k; print("Key set!")
else: print("Demo mode.")

In [ ]:
#@title HITL Agent
import gradio as gr, os, json

TOOLS = [
    {"type": "function", "function": {"name": "lookup_info", "description": "Look up info (safe)", "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}},
    {"type": "function", "function": {"name": "send_email", "description": "Send email (medium risk)", "parameters": {"type": "object", "properties": {"to": {"type": "string"}, "subject": {"type": "string"}, "body": {"type": "string"}}, "required": ["to", "subject", "body"]}}},
    {"type": "function", "function": {"name": "process_refund", "description": "Process refund (high risk)", "parameters": {"type": "object", "properties": {"customer_id": {"type": "string"}, "amount": {"type": "number"}, "reason": {"type": "string"}}, "required": ["customer_id", "amount"]}}}
]

RISK = {"lookup_info": "low", "send_email": "medium", "process_refund": "high"}

def exec_tool(name, args):
    a = json.loads(args) if isinstance(args, str) else args
    if name == "lookup_info": return "Customer: John, Order #12345, $150, Shipped"
    if name == "send_email": return f"Email sent to {a.get('to')}"
    if name == "process_refund": return f"Refund ${a.get('amount')} processed"
    return "Done"

def run_hitl(task, level, auto_under):
    out = [f"# HITL Agent\n**Task:** {task}\n**Approval:** {level}\n---\n"]
    api_key = os.environ.get("OPENAI_API_KEY")
    if not api_key:
        out.append("**[Demo]** Agent looks up info (auto)... sends email (CHECKPOINT)... refund (CHECKPOINT)...\n\n**Summary:** 1 auto, 2 human-approved")
        return "".join(out)
    
    from openai import OpenAI
    client = OpenAI(api_key=api_key)
    msgs = [{"role": "system", "content": "You are customer service. Help with refunds and communication."}, {"role": "user", "content": task}]
    stats = {"auto": 0, "human": 0}
    
    for i in range(5):
        out.append(f"## Step {i+1}\n")
        r = client.chat.completions.create(model="gpt-4o-mini", messages=msgs, tools=TOOLS, max_tokens=300)
        m = r.choices[0].message
        if m.content: out.append(f"**Agent:** {m.content}\n")
        if not m.tool_calls: break
        msgs.append(m)
        for tc in m.tool_calls:
            risk = RISK.get(tc.function.name, "low")
            out.append(f"**Tool:** `{tc.function.name}` (Risk: {risk})\n```\n{tc.function.arguments}\n```\n")
            needs = (level == "All") or (level == "Med+High" and risk in ["medium", "high"]) or (level == "High Only" and risk == "high")
            if tc.function.name == "process_refund":
                amt = json.loads(tc.function.arguments).get("amount", 999)
                if amt <= auto_under: needs = False; out.append(f"*Auto: ${amt} < ${auto_under}*\n")
            if needs: out.append("**CHECKPOINT** - Human approval\n"); stats["human"] += 1
            else: stats["auto"] += 1
            result = exec_tool(tc.function.name, tc.function.arguments)
            out.append(f"**Result:** {result}\n")
            msgs.append({"role": "tool", "tool_call_id": tc.id, "content": result})
        out.append("---\n")
    out.append(f"\n**Summary:** {stats['auto']} auto, {stats['human']} human-approved")
    return "".join(out)

with gr.Blocks(title="HITL Designer", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Human-in-the-Loop Designer\n\nConfigure checkpoints for a **real agent**.")
    with gr.Row():
        with gr.Column(scale=1):
            task = gr.Dropdown(["Customer wants refund for order #12345 ($150)", "Send order update email to customer"], value="Customer wants refund for order #12345 ($150)", label="Task", allow_custom_value=True)
            level = gr.Radio(["All", "Med+High", "High Only", "None"], value="High Only", label="Approval Level")
            auto = gr.Slider(0, 200, value=50, step=25, label="Auto-approve refunds under $")
            btn = gr.Button("Run Agent", variant="primary")
        with gr.Column(scale=2):
            out = gr.Markdown("Configure and run.")
    btn.click(run_hitl, [task, level, auto], out)

In [ ]:
#@title Launch
demo.launch(share=True)